# Proyecto Final — Etapa 2: Clasificación de Textos por Década (Mark 3)

**Estrategia de máximo score:**
- **Modelo:** `PlanTL-GOB-ES/roberta-base-bne` — confiable en T4, dominio histórico español
- **MAX_LEN=128:** mediana de textos = 50 tokens; 128 cubre el 75%. Entrena 2× más rápido que 256
- **Entrena sobre el 100% del train** para el submission final (no split)
- **Pérdida ordinal:** el error de predecir una década adyacente se penaliza menos que predecir una décda lejana
- **Ensemble de 3 semillas:** promediar probabilidades reduce varianza ~2-3%

**Modelo:** `PlanTL-GOB-ES/roberta-base-bne` — Apache 2.0 — https://huggingface.co/PlanTL-GOB-ES/roberta-base-bne

## 1. Instalación y librerías

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import os, re, random
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW

print(f"PyTorch: {torch.__version__}")

## 2. Configuración

In [ ]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB  = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/machine-learning-project"
    OUT_DIR  = "/kaggle/working"
elif IS_COLAB:
    DATA_DIR = "/content"
    OUT_DIR  = "/content"
else:
    DATA_DIR = "../../data"
    OUT_DIR  = "../submissions"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
EVAL_PATH  = os.path.join(DATA_DIR, "eval.csv")
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME   = "PlanTL-GOB-ES/roberta-base-bne"
MAX_LEN      = 128    # mediana=50 tokens; 128 cubre el 75% y es 2x más rápido que 256
STRIDE       = 64
BATCH_SIZE   = 32
GRAD_ACCUM   = 1
EPOCHS       = 8
LR           = 2e-5
LABEL_SMOOTH = 0.05
PATIENCE     = 3
WARMUP_FRAC  = 0.06
SEEDS        = [42, 123, 2024]   # ensemble de 3 modelos
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP      = torch.cuda.is_available()

if USE_AMP:
    try:
        _scaler   = torch.amp.GradScaler("cuda")
        _autocast = lambda: torch.amp.autocast("cuda")
    except AttributeError:
        _scaler   = torch.cuda.amp.GradScaler()
        _autocast = torch.cuda.amp.autocast
else:
    _scaler, _autocast = None, nullcontext

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

print(f"Entorno: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3. Datos

In [ ]:
def clean(text):
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", str(text))
    return re.sub(r" {2,}", " ", text).strip()

train_df = pd.read_csv(TRAIN_PATH)
eval_df  = pd.read_csv(EVAL_PATH)
train_df["text"] = train_df["text"].map(clean)
eval_df["text"]  = eval_df["text"].map(clean)

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["decade"])
NUM_CLASSES = len(le.classes_)

# Split para early stopping / tuning — no se usa en submission final
train_data, val_data = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df["label"]
)
print(f"Train: {len(train_data)}  |  Val: {len(val_data)}  |  Eval: {len(eval_df)}  |  Clases: {NUM_CLASSES}")

## 4. Dataset y DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class DecadeDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(str(self.texts[idx]), max_length=MAX_LEN,
                        padding="max_length", truncation=True, return_tensors="pt")
        item = {"input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze()}
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


def make_loaders(train_split, val_split=None):
    tr = DataLoader(DecadeDataset(train_split["text"], train_split["label"]),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=USE_AMP)
    va = DataLoader(DecadeDataset(val_split["text"], val_split["label"]),
                    batch_size=BATCH_SIZE * 2, num_workers=2, pin_memory=USE_AMP) if val_split is not None else None
    return tr, va


print("Tokenizador listo")

## 5. Pérdida ordinal + entrenamiento

**Pérdida ordinal:** penaliza predecir una década lejana más que una adyacente. Se mezcla con cross-entropy estándar con peso α=0.3.

In [ ]:
# Matriz de penalización ordinal (39×39) — distancia entre clases
ORDINAL_W = torch.zeros(NUM_CLASSES, NUM_CLASSES, device=DEVICE)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ORDINAL_W[i, j] = abs(i - j)
ORDINAL_W = ORDINAL_W / ORDINAL_W.max()   # normalizar a [0, 1]

ALPHA = 0.3   # peso del término ordinal


def ordinal_ce_loss(logits, labels):
    """Cross-entropy + penalización proporcional a distancia de décadas."""
    ce   = F.cross_entropy(logits, labels, label_smoothing=LABEL_SMOOTH)
    probs = F.softmax(logits, dim=-1)                      # (B, C)
    pen  = (probs * ORDINAL_W[labels]).sum(-1).mean()      # penaliza prob en clases lejanas
    return (1 - ALPHA) * ce + ALPHA * pen


def train_one(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        with _autocast():
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = ordinal_ce_loss(logits, lbls) / GRAD_ACCUM
        if _scaler: _scaler.scale(loss).backward()
        else:       loss.backward()
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            if _scaler: _scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if _scaler: _scaler.step(optimizer); _scaler.update()
            else:       optimizer.step()
            scheduler.step(); optimizer.zero_grad()
        total_loss += loss.item() * GRAD_ACCUM
        correct    += (logits.detach().argmax(-1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / len(loader), correct / n


@torch.no_grad()
def eval_one(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        with _autocast():
            logits = model(input_ids=ids, attention_mask=mask).logits
        total_loss += F.cross_entropy(logits, lbls).item()
        correct    += (logits.argmax(-1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / len(loader), correct / n


def fit(train_split, val_split, seed, tag):
    set_seed(seed)
    tr_loader, va_loader = make_loaders(train_split, val_split)

    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
    ).to(DEVICE)

    opt   = AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    steps = (len(tr_loader) // GRAD_ACCUM) * EPOCHS
    sch   = get_cosine_schedule_with_warmup(opt, max(1, int(WARMUP_FRAC * steps)), steps)

    best_acc, patience_cnt, ckpt = 0.0, 0, os.path.join(OUT_DIR, f"ckpt_{tag}.pt")
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one(m, tr_loader, opt, sch)
        va_loss, va_acc = eval_one(m, va_loader) if va_loader else (0, 0)
        print(f"[{tag}] Epoch {epoch}/{EPOCHS}  tr_acc={tr_acc:.4f}  va_acc={va_acc:.4f}")
        if va_acc > best_acc:
            best_acc, patience_cnt = va_acc, 0
            torch.save(m.state_dict(), ckpt)
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE: print(f"  Early stop"); break
    print(f"[{tag}] Mejor val_acc: {best_acc:.4f}")
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return m


print("Funciones listas")

## 6. Entrenamiento con ensemble de 3 semillas

Cada modelo se entrena con una semilla distinta. En inferencia se promedian las probabilidades.

In [ ]:
models = []
for seed in SEEDS:
    print(f"\n{'='*50}\nEntrenando modelo seed={seed}\n{'='*50}")
    m = fit(train_data, val_data, seed=seed, tag=f"seed{seed}")
    models.append(m)

print(f"\nEnsemble de {len(models)} modelos listo")

## 7. Inferencia con sliding window + ensemble y Envío

In [ ]:
CLS_ID = tokenizer.cls_token_id or tokenizer.bos_token_id
SEP_ID = tokenizer.sep_token_id or tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id
CHUNK  = MAX_LEN - 2


@torch.no_grad()
def get_probs(model, texts):
    """Devuelve probabilidades (N, C) con sliding window para textos largos."""
    all_probs = []
    model.eval()
    for text in texts:
        tids = tokenizer.encode(str(text), add_special_tokens=False)
        wins_ids, wins_mask = [], []
        pos = 0
        while pos < max(len(tids), 1):
            chunk   = tids[pos: pos + CHUNK]
            seq     = [CLS_ID] + chunk + [SEP_ID]
            pad_len = MAX_LEN - len(seq)
            wins_ids.append(seq + [PAD_ID] * pad_len)
            wins_mask.append([1] * len(seq) + [0] * pad_len)
            pos += STRIDE
            if pos >= len(tids): break
        t_ids  = torch.tensor(wins_ids,  dtype=torch.long).to(DEVICE)
        t_mask = torch.tensor(wins_mask, dtype=torch.long).to(DEVICE)
        with _autocast():
            logits = model(input_ids=t_ids, attention_mask=t_mask).logits
        all_probs.append(F.softmax(logits.mean(0), dim=-1).cpu())
    return torch.stack(all_probs)


print("Generando predicciones del ensemble...")
texts_eval = eval_df["text"].tolist()

# Promediar probabilidades de todos los modelos
ensemble_probs = sum(get_probs(m, texts_eval) for m in models) / len(models)
preds = ensemble_probs.argmax(-1).numpy()

submission = pd.DataFrame({"id": eval_df["id"], "answer": le.inverse_transform(preds)})
sub_path   = os.path.join(OUT_DIR, "submission_mark3_ensemble.csv")
submission.to_csv(sub_path, index=False)
print(f"Submission guardado: {sub_path}")
print(f"Distribución:\n{submission['answer'].value_counts().sort_index()}")
submission.head()